Background Tests

The purpose of this notebook is to do the aperture and box background tests for Euclid data

In [3]:
# | hide
from nbdev import show_doc

In [1]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import PercentileInterval, AsinhStretch, ImageNormalize
from astropy.nddata import Cutout2D
from astropy.cosmology import FlatLambdaCDM
from photutils.background import Background2D, MedianBackground
from astropy.convolution import convolve
from photutils.segmentation import make_2dgaussian_kernel, detect_sources, deblend_sources
from photutils.aperture import CircularAnnulus, CircularAperture, aperture_photometry, EllipticalAnnulus, EllipticalAperture
#from ellipse import LsqEllipse
from matplotlib.patches import Ellipse
from scipy import interpolate
from scipy.interpolate import interp1d

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
import os
import sys
from photutils.aperture import ApertureStats

This function generates randomly placed boxes.  Then measures the median per pixel value within each box.  Lastly it returns the standard deviation of those medians across all boxes.  The needed inputs are the masked file that you are running this on and the array of box sizes.  Note the input size is 1/2 of the length of the box.  

In [6]:
def background_boxes(file,box_sizes):
    input = file
    hdu=fits.open(input)[0]
    
    imx,imy=np.shape(hdu.data)
    print (imx,imy)

    a=np.random.randint(np.max(box_sizes)+1,imx-np.max(box_sizes)-1,1000)
    b=np.random.randint(np.max(box_sizes)+1,imy-np.max(box_sizes)-1,1000)
    
    sizes=box_sizes
    sizes=np.array(sizes)
    std_dev_flux=np.zeros(len(sizes))
    for p in range(0,len(sizes)):
        print (p)
        zero_percentage=np.zeros(len(a))
        median_values=[]
    
        if sizes[p] == 0:
            for i in range(0,len(a)):
                c1=a[i]
                c2=b[i]
        
                if hdu.data[c1][c2] !=0.0:
                    median_values.append(hdu.data[c1][c2])
            
        if sizes[p] > 0:
        
            for i in range(0,len(a)):
                c1=np.arange(a[i]-sizes[p],a[i]+sizes[p]+1,1)
                c2=np.arange(b[i]-sizes[p],b[i]+sizes[p]+1,1)
        
                subset_med=np.median(hdu.data[c1[0]:c1[-1],c2[0]:c2[-1]])
                zeros=(len(np.where(hdu.data[c1[0]:c1[-1],c2[0]:c2[-1]]==0.0)[0]))
                total=(2.*sizes[p])**2
                ratio=zeros*1.0/total
                zero_percentage[i]=(ratio)
                if total > 100 and subset_med < 1.0:
                    if ratio < 0.4:
                        median_values.append(subset_med)
                if total <= 100 and subset_med < 1.0:
                    if ratio ==0.0:
                        median_values.append(subset_med)
        median_values=np.array(median_values)
        std_dev_flux[p]=np.sqrt(np.var(median_values))
        zero_percentage=np.array(zero_percentage)
    return (std_dev_flux)

This function does a similar test, but with aperture rings.  First we assign random points throughout our images, then we meausre the median pixel flux within an aperture centered on those images.  Lastly we retun the standard deviation of those medians.  

In [7]:
def background_rings(file,maskfile,radius_size):
    input = file
    
    hdu=fits.open(input)[0]
    im_test=hdu.data
    
    imx,imy=np.shape(hdu.data)

    a=np.random.randint(np.max(radius_size)+1,imx-np.max(radius_size)-1,1000)
    b=np.random.randint(np.max(radius_size)+1,imy-np.max(radius_size)-1,1000)
    
    mask = maskfile
    
    hdumask=fits.open(mask)[0]
    
    masked_final = hdumask.data
    mask_final=masked_final.astype(bool)
    
    std_dev_flux1=np.zeros(len(radius_size)-1)
    
    radius_test=radius_size
    
    for h in range(0,len(radius_test)-1):
        print (h)
        all_fluxes=[]
        ICL_flux_circleb=np.zeros(len(a))
        flux_circleb=np.zeros(len(a))
        masked_fraction=np.zeros(len(a))
        mask_flux=np.zeros(len(a))
        area2=np.zeros(len(a))
        pixel_to_arcsec=(0.3)

        for j in range(len(a)):
            coordinates_test=(a[j],b[j])
            if h == 0:
                aper_circle = CircularAperture(coordinates_test, (radius_test[h]))
                area=aper_circle.area
                aperstats=ApertureStats(im_test,aper_circle,mask=mask_final)
                area2[j]=(aperstats.sum_aper_area.value)
                ICL_flux_circleb[j]=aperstats.mean
            if h > 0:
                aper_circle = CircularAnnulus(coordinates_test, (radius_test[h]), ((radius_test[h+1])))
                area=aper_circle.area
                aperstats=ApertureStats(im_test,aper_circle,mask=mask_final)
                area2[j]=(aperstats.sum_aper_area.value)
                ICL_flux_circleb[j]=aperstats.mean
            
            masked_fraction[j]=area2[j]/area
            if masked_fraction[j] > 0.6:
                all_fluxes.append(ICL_flux_circleb[j])
        all_fluxes=np.array(all_fluxes)
        std_dev_flux1[h]=np.sqrt(np.var(all_fluxes))
    
    return (std_dev_flux1)
